<a href="https://colab.research.google.com/github/xhesikam/shakespeare_ai/blob/main/Artificial_Intelligence_Shakespeare_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import torch.nn as nn
from torch.nn import functional as F
from datetime import datetime

1. Scale the model up: I will do this by increasing training time, batch_size, block_size, n_embd, n_head, n_layer: DONE
2. Turn on the dropout mechanism: DONE dropout is 0.5
3. Train the model on GPU. All moved to GPU: DONE
- Record the training time (start, finish, and time elapsed): DONE
- Generate 5000 characters from the model and analyze its performance: DONE

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print("device: ", device)

device:  cuda


In [ ]:
#variables
batch_size = 32
max_iters = 10000
learning_rate = 1e-3
block_size = 64
eval_interval = 100
eval_iters = 200
n_embd = 128
n_head = 8
n_layer = 8
torch.manual_seed(1334)

In [ ]:
# Load data
! wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

--2025-12-22 23:07:41--  https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1115394 (1.1M) [text/plain]
Saving to: ‘input.txt.1’

input.txt.1         100%[===================>]   1.06M  --.-KB/s    in 0.03s   

2025-12-22 23:07:42 (30.8 MB/s) - ‘input.txt.1’ saved [1115394/1115394]



In [ ]:
chars = sorted(list(set(text)))
vocab_size = len(chars)
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join([itos[i] for i in l])
print("Vocabulary Size:", vocab_size)
print("Mapping:")
print(stoi)

Vocabulary Size: 65
Mapping:
{'\n': 0, ' ': 1, '!': 2, '$': 3, '&': 4, "'": 5, ',': 6, '-': 7, '.': 8, '3': 9, ':': 10, ';': 11, '?': 12, 'A': 13, 'B': 14, 'C': 15, 'D': 16, 'E': 17, 'F': 18, 'G': 19, 'H': 20, 'I': 21, 'J': 22, 'K': 23, 'L': 24, 'M': 25, 'N': 26, 'O': 27, 'P': 28, 'Q': 29, 'R': 30, 'S': 31, 'T': 32, 'U': 33, 'V': 34, 'W': 35, 'X': 36, 'Y': 37, 'Z': 38, 'a': 39, 'b': 40, 'c': 41, 'd': 42, 'e': 43, 'f': 44, 'g': 45, 'h': 46, 'i': 47, 'j': 48, 'k': 49, 'l': 50, 'm': 51, 'n': 52, 'o': 53, 'p': 54, 'q': 55, 'r': 56, 's': 57, 't': 58, 'u': 59, 'v': 60, 'w': 61, 'x': 62, 'y': 63, 'z': 64}


In [ ]:
data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9*len(data))
train_data = data[:n]
val_data = data[n:]
print(len(train_data), len(val_data))

1003854 111540


In [ ]:
def get_batch(split):
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y

In [ ]:
@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

In [ ]:
dropout = 0.5

In [ ]:
class Head(nn.Module):

    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B,T,C = x.shape
        k = self.key(x)
        q = self.query(x)
        v = self.value(x)

        wei = q @ k.transpose(-2,-1) * C**-0.5
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        wei = F.softmax(wei, dim=-1)
        wei = self.dropout(wei)
        out = wei @ v
        return out

In [ ]:
class MultiHeadAttention(nn.Module):

    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.proj(out)
        out = self.dropout(out)
        return out

In [ ]:
class FeedForward(nn.Module):
    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)

In [ ]:
class Block(nn.Module):

    def __init__(self, n_embd, n_head):
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedForward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

In [ ]:
class SelfAttentionModel(nn.Module):

    def __init__(self):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head=n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape

        tok_emb = self.token_embedding_table(idx)
        pos_emb = self.position_embedding_table(torch.arange(T, device=device)) # (T,C)
        x = tok_emb + pos_emb
        x = self.blocks(x)
        x = self.ln_f(x)
        logits = self.lm_head(x)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -block_size:]
            logits, loss = self(idx_cond)
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx

In [ ]:
model = SelfAttentionModel()
m = model.to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)
training_start = datetime.now()
print("Training started: ",training_start)
for iter in range(max_iters):
    if iter % eval_interval == 0 or iter == max_iters - 1:
        losses = estimate_loss()
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    xb, yb = get_batch('train')

    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
training_end = datetime.now()
print("Training ended: ",training_end)
print("Training elapsed: ",training_end - training_start)

Training started:  2025-12-22 23:08:02.354633
step 0: train loss 4.2767, val loss 4.2731
step 100: train loss 2.5071, val loss 2.5127
step 200: train loss 2.4269, val loss 2.4366
step 300: train loss 2.3640, val loss 2.3793
step 400: train loss 2.2859, val loss 2.3045
step 500: train loss 2.2176, val loss 2.2560
step 600: train loss 2.1564, val loss 2.1883
step 700: train loss 2.1042, val loss 2.1512
step 800: train loss 2.0519, val loss 2.0978
step 900: train loss 2.0049, val loss 2.0603
step 1000: train loss 1.9801, val loss 2.0566
step 1100: train loss 1.9302, val loss 2.0117
step 1200: train loss 1.9063, val loss 1.9950
step 1300: train loss 1.8797, val loss 1.9804
step 1400: train loss 1.8572, val loss 1.9614
step 1500: train loss 1.8191, val loss 1.9454
step 1600: train loss 1.8014, val loss 1.9232
step 1700: train loss 1.7871, val loss 1.9089
step 1800: train loss 1.7728, val loss 1.9137
step 1900: train loss 1.7523, val loss 1.9055
step 2000: train loss 1.7478, val loss 1.8933


In [ ]:
#generate 5000 characters
context = torch.zeros((1, 1), dtype=torch.long, device=device)
print(decode(m.generate(context, max_new_tokens=5000)[0].tolist()))


;
Ay, look come, these true ages you fair.

Senation:
Pleal 'tidio say, lisperabl in to the kint,
Give pain'd set our vurkely deshren hid incall'd
With lats their queen own, the sol whom with.

BUCKINGHAM:
It was pasts oth us?

KING RICHARD I:
The procese the Helgd's broke, To the he both den
Some withstran shine foetast as my Monuh
If Griek Richard it.
Till sayst the mest, Though this, thee should and stay.
Whithen there this hath stond be'er;
Bunites 'tis that since wn this deed at her with now
'Tis size is thy purp stubb'd by forial edgzed reposed,
Stame and my brother: u speak, my lord,
O hearts mights no more day, shoulding die
cons, myself; one years the pair of Norabans!

No:
Even-sutledling speak but at him wa's my Exewills:
And God yet, no make vights heven referites
In him Lord ssunabehur did returepred his sajudies!

GREMIO:
But why 'll I have my can!

YUS:
God, that Land, for my sin good,
For so comest, think the comfort werve,
So is set these aughter, bne'er bluts
he stat

The model seems to be perfoming pretty well. It seems to know about capitalzition, and puncauation. There are a lot of real words created. It could perfrom better and there are somethings it could do better, but overal this is pretty impressive. The model was training for around 30 minutes for 10000 iterations. There seems to be a bit of overfeeding, but not too much. Overall, the model performs pretty good. There are some issues, but again it is still impressive how many real words it was able to create. It learned capitialztion  and puncauation pretty well. It has been upscaled by making the training session 10000 iterations and by increasing the sizes of these varibales: batch_size, block_size, n_embd, n_head, n_layer.